# Combat PPO training

**Snapshots:** JSON files under **``<REPO_ROOT>/snapshots/*.json``** (training pool). A demo file may live at ``temp/snapshot_no_jokers_level1.json`` (gitignored); load with ``env.snapshot_io.load_snapshot_pool_from_json_dir`` or ``load_snapshot``. If the pool directory is missing or empty, the notebook falls back to the in-code demo pool.

**Overrides:** ``BALATRO_AGENT_REPO`` = repo root (set this on Colab). ``BALATRO_SNAPSHOT_JSON_DIR`` = alternate directory of ``*.json`` (defaults to ``<REPO_ROOT>/snapshots``).

**Checkpoints:** periodic saves under ``REPO_ROOT/checkpoints/`` (model, optimizer, iteration, ``PPOConfig``). Set ``RESUME_CHECKPOINT`` or env ``BALATRO_RESUME_CKPT`` to resume; LR scheduler is recreated from ``cfg`` and iteration (not stored in the file).

**Logging:** Duke-style line (last 50 episodes): ``reward``, ``len``, ``win%``, ``pg``, ``vf``, ``ent``, ``episodes``.

**Colab (Duke-style):** the next cell mounts Google Drive (``drive.mount("/content/drive")``). Repo default: ``/content/drive/MyDrive/Balatro_Agent_Project`` (override with ``BALATRO_AGENT_REPO``). Use ``BALATRO_VEC_BACKEND=sync`` if async multiprocessing flakes.


In [1]:
import os
import sys

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass  # local run: skip mount; set REPO_DIR to your clone

# Your Drive clone (must match folder name on Disk exactly).
REPO_DIR = "/content/drive/MyDrive/balatro-agent-project"
REPO_ROOT = REPO_DIR  # alias for checkpoint paths later in the notebook

os.chdir(REPO_DIR)

for p in (
    REPO_DIR,
    os.path.join(REPO_DIR, "balatro_lite_gym"),
):
    if p not in sys.path:
        sys.path.insert(0, p)

SNAPSHOT_JSON_DIR = os.environ.get(
    "BALATRO_SNAPSHOT_JSON_DIR",
    os.path.join(REPO_ROOT, "snapshots"),
)
VEC_BACKEND = os.environ.get("BALATRO_VEC_BACKEND", "async").strip().lower()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import numpy as np
import torch
from torch.distributions import Categorical

from env.lite_combat_env import (
    VecRolloutBuffer,
    compute_gae_vectorized,
    dict_to_tensors,
    make_vec,
)
from agent.model import CombatPPOAgent
from agent.ppo import get_card_mask, mask_logits, ppo_update
from agent.ppo_config import PPOConfig
from env.snapshot_io import load_snapshot_pool_from_json_dir


In [3]:
# Load every ``*.json`` under SNAPSHOT_JSON_DIR (default: ``<REPO_ROOT>/snapshots``).
snapshot_pool = load_snapshot_pool_from_json_dir(SNAPSHOT_JSON_DIR)

print("pool size:", len(snapshot_pool))


pool size: 1


In [4]:
cfg = PPOConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
start_iteration = 1
# Optional: override after env, e.g. RESUME_CHECKPOINT = os.path.join(REPO_ROOT, "checkpoints/combat_ppo_iter_100.pt")
RESUME_CHECKPOINT = os.environ.get("BALATRO_RESUME_CKPT") or None
if isinstance(RESUME_CHECKPOINT, str) and not RESUME_CHECKPOINT.strip():
    RESUME_CHECKPOINT = None
CKPT_EVERY = 100
ckpt = None

if RESUME_CHECKPOINT:
    try:
        ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
    except TypeError:
        ckpt = torch.load(RESUME_CHECKPOINT, map_location=device)
    cfg = ckpt["config"]
    start_iteration = int(ckpt["iteration"]) + 1
    print(f"Resumed from {RESUME_CHECKPOINT!r} — next iteration {start_iteration}")

assert (cfg.rollout_steps * cfg.num_envs) % cfg.num_minibatches == 0, (
    f"rollout_steps * num_envs must divide num_minibatches; got "
    f"{cfg.rollout_steps} * {cfg.num_envs} vs {cfg.num_minibatches}"
)

agent = CombatPPOAgent(
    d_model=cfg.d_model,
    nhead=cfg.nhead,
    dim_ff=cfg.dim_ff,
    dropout=cfg.dropout,
).to(device)
optimizer = torch.optim.Adam(agent.parameters(), lr=cfg.lr, eps=1e-5)

if ckpt is not None:
    agent.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])

if cfg.use_lr_scheduler:
    sch_last = max(-1, start_iteration - 2)
    lr_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1.0,
        end_factor=cfg.lr_min_ratio,
        total_iters=max(1, cfg.max_iterations),
        last_epoch=sch_last,
    )
else:
    lr_scheduler = None

if VEC_BACKEND not in ("async", "sync"):
    raise ValueError(f"VEC_BACKEND must be 'async' or 'sync', got {VEC_BACKEND!r}")

vec = make_vec(
    snapshot_pool,
    n=cfg.num_envs,
    base_seed=0,
    backend=VEC_BACKEND,
)
buffer = VecRolloutBuffer(cfg.rollout_steps, cfg.num_envs, device)

print("device:", device, "| vector backend:", VEC_BACKEND)


device: cuda | vector backend: async


In [5]:
def _per_env_combat_won(infos, num_envs: int) -> np.ndarray:
    if not isinstance(infos, dict) or "combat_won" not in infos:
        return np.zeros(num_envs, dtype=bool)
    cw = np.asarray(infos["combat_won"])
    if cw.dtype == object:
        row = [bool(x) for x in cw.tolist()[:num_envs]]
        return np.pad(row, (0, max(0, num_envs - len(row))), constant_values=False)
    flat = cw.astype(bool).reshape(-1)[:num_envs]
    if flat.size < num_envs:
        out = np.zeros(num_envs, dtype=bool)
        out[: flat.size] = flat
        return out
    return flat


def _wins_from_snapshot_fallback(infos, num_envs: int, dones_np: np.ndarray) -> np.ndarray:
    out = np.zeros(num_envs, dtype=bool)
    if not isinstance(infos, dict) or "snapshot" not in infos or not dones_np.any():
        return out
    snaps = infos["snapshot"]
    if isinstance(snaps, (list, tuple)):
        seq = list(snaps)
    else:
        seq = [snaps] * num_envs
    for i in np.where(dones_np)[0]:
        if i < len(seq):
            s = seq[i]
            out[i] = bool(s.current_score >= s.target_score)
    return out


def _episode_wins(infos, num_envs: int, dones_np: np.ndarray) -> np.ndarray:
    w = _per_env_combat_won(infos, num_envs)
    if isinstance(infos, dict) and "combat_won" not in infos:
        w = _wins_from_snapshot_fallback(infos, num_envs, dones_np)
    return w


In [6]:
obs_np, _ = vec.reset(seed=None)
ep_returns: list[float] = []
ep_lengths: list[int] = []
ep_wins: list[bool] = []
running_rewards = np.zeros(cfg.num_envs)
running_lens = np.zeros(cfg.num_envs, dtype=int)

for iteration in range(start_iteration, cfg.max_iterations + 1):
    agent.eval()
    for t in range(cfg.rollout_steps):
        obs_t = dict_to_tensors(obs_np, device)
        card_mask = get_card_mask(obs_t)

        with torch.no_grad():
            sel_logits, exec_logits, value = agent(obs_t)

        sel_logits = mask_logits(sel_logits, card_mask)
        sel_dist = Categorical(logits=sel_logits)
        exec_dist = Categorical(logits=exec_logits)

        card_sels = sel_dist.sample()
        executions = exec_dist.sample()

        log_probs = (
            sel_dist.log_prob(card_sels).sum(dim=-1)
            + exec_dist.log_prob(executions)
        )

        actions_np = np.concatenate(
            [
                card_sels.cpu().numpy(),
                executions.cpu().numpy()[:, None],
            ],
            axis=-1,
        )

        next_obs_np, rewards_np, terms, truncs, infos = vec.step(actions_np)
        dones_np = np.logical_or(terms, truncs)

        buffer.store_step(
            t,
            obs_t,
            card_sels,
            executions,
            log_probs,
            value.squeeze(-1),
            rewards_np,
            dones_np,
        )

        running_rewards += rewards_np
        running_lens += 1
        ce_vec = _episode_wins(infos, cfg.num_envs, dones_np)
        for i in np.where(dones_np)[0]:
            ep_returns.append(float(running_rewards[i]))
            ep_lengths.append(int(running_lens[i]))
            ep_wins.append(bool(ce_vec[i]))
            running_rewards[i] = 0.0
            running_lens[i] = 0

        obs_np = next_obs_np

    with torch.no_grad():
        _, _, next_vals = agent(dict_to_tensors(obs_np, device))
        next_vals = next_vals.squeeze(-1)

    advantages, returns = compute_gae_vectorized(
        buffer.rewards,
        buffer.values,
        next_vals,
        buffer.dones,
        cfg.gamma,
        cfg.gae_lambda,
    )

    flat_obs, flat_card_sels, flat_execs, flat_lp = buffer.flatten()
    flat_adv = advantages.reshape(-1)
    flat_ret = returns.reshape(-1)

    agent.train()
    losses = ppo_update(
        agent,
        optimizer,
        flat_obs,
        flat_card_sels,
        flat_execs,
        flat_lp,
        flat_adv,
        flat_ret,
        cfg,
    )

    if lr_scheduler is not None:
        lr_scheduler.step()

    if iteration % CKPT_EVERY == 0:
        ckpt_dir = os.path.join(REPO_ROOT, "checkpoints")
        os.makedirs(ckpt_dir, exist_ok=True)
        periodic_path = os.path.join(ckpt_dir, f"combat_ppo_iter_{iteration}.pt")
        torch.save(
            {
                "model_state_dict": agent.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "config": cfg,
                "iteration": iteration,
            },
            periodic_path,
        )
        print(f"Periodic checkpoint saved to {periodic_path}")

    if iteration % 5 == 0 or iteration == 1:
        recent_r = np.mean(ep_returns[-50:]) if ep_returns else 0.0
        recent_len = np.mean(ep_lengths[-50:]) if ep_lengths else 0.0
        recent_win = 100.0 * np.mean(ep_wins[-50:]) if ep_wins else 0.0
        print(
            f"[iter {iteration:4d}]  "
            f"reward={recent_r:+.2f}  len={recent_len:.1f}  "
            f"win={recent_win:.1f}%  "
            f"pg={losses['pg_loss']:.4f}  "
            f"vf={losses['value_loss']:.4f}  "
            f"ent={losses['entropy']:.4f}  "
            f"episodes={len(ep_returns)}"
        )


[iter    1]  reward=-0.79  len=9.2  win=0.0%  pg=0.0395  vf=2.7900  ent=6.0080  episodes=91
[iter    5]  reward=+0.98  len=6.1  win=0.0%  pg=-0.0140  vf=0.2892  ent=5.7535  episodes=713
[iter   10]  reward=+1.07  len=5.7  win=0.0%  pg=-0.0075  vf=0.1775  ent=5.4504  episodes=1595
[iter   15]  reward=+1.01  len=5.5  win=0.0%  pg=-0.0032  vf=0.2090  ent=5.3627  episodes=2529
[iter   20]  reward=+1.09  len=5.4  win=0.0%  pg=0.0061  vf=0.1766  ent=5.2542  episodes=3486
[iter   25]  reward=+1.24  len=5.2  win=2.0%  pg=-0.0055  vf=0.1326  ent=5.2371  episodes=4454
[iter   30]  reward=+1.23  len=5.2  win=0.0%  pg=-0.0044  vf=0.1358  ent=5.3302  episodes=5425
[iter   35]  reward=+1.20  len=5.2  win=0.0%  pg=0.0004  vf=0.1296  ent=5.2720  episodes=6402
[iter   40]  reward=+1.15  len=5.2  win=2.0%  pg=0.0001  vf=0.1539  ent=5.1754  episodes=7376
[iter   45]  reward=+1.11  len=5.3  win=0.0%  pg=0.0001  vf=0.1532  ent=5.3125  episodes=8353
[iter   50]  reward=+1.26  len=5.1  win=0.0%  pg=-0.0021  

KeyboardInterrupt: 

In [ ]:
vec.close()
